# IPL T20 Graph Demo

**Pipeline:** Synthetic data → Parquet → GCS → FalkorDB → Cypher

Graph shape:
- **Team** nodes, **Player** nodes, **Game** nodes
- **PLAYS_FOR**, **PLAYED_IN**, **WON** edges

In [ ]:
# Step 1 — Setup
import requests, json, time, os
import pandas as pd
from pathlib import Path

API = "http://graph-olap-control-plane:8080"
GCS = "http://fake-gcs-local:4443"
H   = {"X-Username": "demo@example.com", "X-User-Role": "admin", "Content-Type": "application/json"}

def get(path):        r = requests.get(f"{API}{path}", headers=H);   r.raise_for_status(); return r.json()
def post(path, body): r = requests.post(f"{API}{path}", headers=H, json=body); r.raise_for_status(); return r.json()
def delete(path):     r = requests.delete(f"{API}{path}", headers=H); r.raise_for_status(); return r.json()

print(requests.get(f"{API}/health").json())

In [ ]:
# Step 2 — Create Sample Data
teams = pd.DataFrame([{"team_id":1,"name":"Mumbai Indians"},{"team_id":2,"name":"Chennai Super Kings"},{"team_id":3,"name":"Royal Challengers"},{"team_id":4,"name":"Kolkata Knight Riders"},{"team_id":5,"name":"Delhi Capitals"},{"team_id":6,"name":"Rajasthan Royals"},{"team_id":7,"name":"Sunrisers Hyderabad"},{"team_id":8,"name":"Punjab Kings"}])
players = pd.DataFrame([{"player_id":1,"name":"Rohit Sharma","role":"Bat"},{"player_id":2,"name":"MS Dhoni","role":"WK"},{"player_id":3,"name":"Virat Kohli","role":"Bat"},{"player_id":4,"name":"Andre Russell","role":"AR"},{"player_id":5,"name":"Rishabh Pant","role":"WK"},{"player_id":6,"name":"Jos Buttler","role":"Bat"},{"player_id":7,"name":"Bhuvneshwar Kumar","role":"Bowl"},{"player_id":8,"name":"Shikhar Dhawan","role":"Bat"},{"player_id":9,"name":"Jasprit Bumrah","role":"Bowl"},{"player_id":10,"name":"Ravindra Jadeja","role":"AR"},{"player_id":11,"name":"KL Rahul","role":"Bat"},{"player_id":12,"name":"Hardik Pandya","role":"AR"},{"player_id":13,"name":"Suryakumar Yadav","role":"Bat"},{"player_id":14,"name":"Rashid Khan","role":"Bowl"},{"player_id":15,"name":"Kagiso Rabada","role":"Bowl"}])
games = pd.DataFrame([{"match_id":1,"venue":"Wankhede"},{"match_id":2,"venue":"Chepauk"},{"match_id":3,"venue":"Chinnaswamy"},{"match_id":4,"venue":"Eden Gardens"},{"match_id":5,"venue":"Wankhede"}])
plays_for = pd.DataFrame([{"player_id":1,"team_id":1},{"player_id":9,"team_id":1},{"player_id":12,"team_id":1},{"player_id":13,"team_id":1},{"player_id":2,"team_id":2},{"player_id":10,"team_id":2},{"player_id":3,"team_id":3},{"player_id":4,"team_id":4},{"player_id":5,"team_id":5},{"player_id":15,"team_id":5},{"player_id":6,"team_id":6},{"player_id":7,"team_id":7},{"player_id":14,"team_id":7},{"player_id":8,"team_id":8},{"player_id":11,"team_id":8}])
played_in = pd.DataFrame([{"player_id":1,"match_id":1},{"player_id":2,"match_id":1},{"player_id":9,"match_id":1},{"player_id":10,"match_id":1},{"player_id":2,"match_id":2},{"player_id":3,"match_id":2},{"player_id":3,"match_id":3},{"player_id":4,"match_id":3},{"player_id":4,"match_id":4},{"player_id":5,"match_id":4},{"player_id":15,"match_id":4},{"player_id":1,"match_id":5},{"player_id":6,"match_id":5},{"player_id":12,"match_id":5}])
won = pd.DataFrame([{"team_id":1,"match_id":1},{"team_id":2,"match_id":2},{"team_id":3,"match_id":3},{"team_id":4,"match_id":4},{"team_id":1,"match_id":5}])
print(f"Teams:{len(teams)} Players:{len(players)} Games:{len(games)} PLAYS_FOR:{len(plays_for)} PLAYED_IN:{len(played_in)} WON:{len(won)}")

In [ ]:
# Step 3 — Write Parquet
base = Path("/tmp/ipl-graph")
for p in ["nodes/Team","nodes/Player","nodes/Game","edges/PLAYS_FOR","edges/PLAYED_IN","edges/WON"]:
    (base / p).mkdir(parents=True, exist_ok=True)
teams.to_parquet(base/"nodes/Team/data.parquet",index=False)
players.to_parquet(base/"nodes/Player/data.parquet",index=False)
games.to_parquet(base/"nodes/Game/data.parquet",index=False)
plays_for.to_parquet(base/"edges/PLAYS_FOR/data.parquet",index=False)
played_in.to_parquet(base/"edges/PLAYED_IN/data.parquet",index=False)
won.to_parquet(base/"edges/WON/data.parquet",index=False)
print("✅ Parquet written")

In [ ]:
# Step 4 — Create Mapping
mapping = post("/api/mappings", {
    "name": "IPL T20 Cricket",
    "node_definitions": [
        {"label":"Team","sql":"SELECT team_id,name FROM teams","primary_key":{"name":"team_id","type":"INT64"},"properties":[{"name":"name","type":"STRING"}]},
        {"label":"Player","sql":"SELECT player_id,name,role FROM players","primary_key":{"name":"player_id","type":"INT64"},"properties":[{"name":"name","type":"STRING"},{"name":"role","type":"STRING"}]},
        {"label":"Game","sql":"SELECT match_id,venue FROM games","primary_key":{"name":"match_id","type":"INT64"},"properties":[{"name":"venue","type":"STRING"}]},
    ],
    "edge_definitions": [
        {"type":"PLAYS_FOR","sql":"SELECT player_id,team_id FROM plays_for","from_node":"Player","to_node":"Team","from_key":"player_id","to_key":"team_id","properties":[]},
        {"type":"PLAYED_IN","sql":"SELECT player_id,match_id FROM played_in","from_node":"Player","to_node":"Game","from_key":"player_id","to_key":"match_id","properties":[]},
        {"type":"WON","sql":"SELECT team_id,match_id FROM won","from_node":"Team","to_node":"Game","from_key":"team_id","to_key":"match_id","properties":[]},
    ],
})
mapping_id = mapping["data"]["id"]
print(f"✅ Mapping created: id={mapping_id}")

In [ ]:
# Step 5 — Create Instance + Mark Snapshot Ready
import psycopg2

instance = post("/api/instances", {"mapping_id": mapping_id, "wrapper_type": "falkordb", "name": "IPL T20 Instance", "ttl": "PT4H"})
instance_id = instance["data"]["id"]
snapshot_id = instance["data"].get("snapshot_id") or instance["data"].get("pending_snapshot_id")
print(f"✅ Instance created: id={instance_id}, snapshot_id={snapshot_id}")

BUCKET = "graph-olap-local-dev"
gcs_path = f"demo@example.com/{mapping_id}/v1/{snapshot_id}"

conn = psycopg2.connect(host="postgres", port=5432, dbname="control_plane", user="control_plane", password="control_plane")
conn.autocommit = True
cur = conn.cursor()
cur.execute("UPDATE snapshots SET status='ready', gcs_path=%s WHERE id=%s", (f"gs://{BUCKET}/{gcs_path}", snapshot_id))
cur.execute("DELETE FROM export_jobs WHERE snapshot_id=%s", (snapshot_id,))
cur.close(); conn.close()
print(f"✅ Snapshot {snapshot_id} marked ready")

In [ ]:
# Step 6 — Upload Parquet to GCS
_req = requests.Session()
_req.post(f"{GCS}/storage/v1/b", json={"name": BUCKET}, headers={"Content-Type": "application/json"})
for local, remote in [("nodes/Team/data.parquet",f"{gcs_path}/nodes/Team/data.parquet"),("nodes/Player/data.parquet",f"{gcs_path}/nodes/Player/data.parquet"),("nodes/Game/data.parquet",f"{gcs_path}/nodes/Game/data.parquet"),("edges/PLAYS_FOR/data.parquet",f"{gcs_path}/edges/PLAYS_FOR/data.parquet"),("edges/PLAYED_IN/data.parquet",f"{gcs_path}/edges/PLAYED_IN/data.parquet"),("edges/WON/data.parquet",f"{gcs_path}/edges/WON/data.parquet")]:
    with open(base/local,"rb") as f: data=f.read()
    resp=_req.post(f"{GCS}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={remote}",data=data,headers={"Content-Type":"application/octet-stream"})
    print(f"  {'✅' if resp.status_code in (200,201) else '❌'} {local}")

In [ ]:
# Step 7 — Wait for Instance Running
print("Polling...")
start = time.time()
while True:
    elapsed = int(time.time() - start)
    inst = get(f"/api/instances/{instance_id}")
    status = inst["data"]["status"]
    print(f"  [{elapsed:3d}s] instance={status}")
    if status == "running": break
    if status in ("stopped","failed"): raise RuntimeError(f"Instance failed: {status}")
    time.sleep(10)

pod_name = inst["data"].get("pod_name", "")
WRAPPER = f"http://{pod_name}:8000"
print(f"\n🎉 IPL T20 graph is running! Wrapper: {WRAPPER}")

## Step 8 — Query the Graph

In [ ]:
def cypher(q):
    r = requests.post(f"{WRAPPER}/query", json={"query": q}); r.raise_for_status()
    d = r.json()
    return pd.DataFrame(d["rows"], columns=d["columns"])

# Node counts
cypher("MATCH (n) RETURN labels(n)[0] as type, count(n) as count ORDER BY count DESC")

In [ ]:
# Players per team
cypher("MATCH (p:Player)-[:PLAYS_FOR]->(t:Team) RETURN t.name as team, collect(p.name) as players ORDER BY team")

In [ ]:
# Teams with most wins
cypher("MATCH (t:Team)-[:WON]->(g:Game) RETURN t.name as team, count(g) as wins, collect(g.venue) as venues ORDER BY wins DESC")

In [ ]:
# Multi-hop: Players who played in games their team won
cypher("MATCH (p:Player)-[:PLAYS_FOR]->(t:Team)-[:WON]->(g:Game)<-[:PLAYED_IN]-(p) RETURN p.name as player, t.name as team, g.venue as venue ORDER BY team")

In [ ]:
# Cleanup (uncomment to terminate)
# delete(f"/api/instances/{instance_id}")
# print(f"Instance {instance_id} terminated")